<a href="https://colab.research.google.com/github/MithunSrinivas28/wafer-defect-ai/blob/main/Wafer_detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Mount Google Drive**

In [5]:
import tensorflow as tf
tf.keras.backend.clear_session()


In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Load Dataset Using TensorFlow

In [7]:
TRAIN_PATH = "/content/drive/MyDrive/Datasets/train"
TEST_PATH  = "/content/drive/MyDrive/Datasets/test"


In [8]:
#import os
#import shutil
#import random
#import math

# -------- CHANGE ONLY IF NEEDED --------
#TRAIN_DIR = "/content/drive/MyDrive/Datasets/train"
T#EST_DIR  = "/content/drive/MyDrive/Datasets/test"

# --------------------------------------

#os.makedirs(TEST_DIR, exist_ok=True)

#for class_name in os.listdir(TRAIN_DIR):

    #train_class_path = os.path.join(TRAIN_DIR, class_name)
    #test_class_path = os.path.join(TEST_DIR, class_name)

    #if not os.path.isdir(train_class_path):
        #continue

    #os.makedirs(test_class_path, exist_ok=True)

    #images = [img for img in os.listdir(train_class_path)
              #if img.lower().endswith((".jpg", ".jpeg", ".png"))]

    #total = len(images)
    #test_count = math.ceil(0.20 * total)   # 20% for test
    #train_count = total - test_count       # remaining 80%

    #print(f"\nClass: {class_name}")
    #print(f"Total images: {total}")
    #print(f"Train target: {train_count}")
    #print(f"Test target:  {test_count}")

    #selected_for_test = random.sample(images, test_count)

    #for img in selected_for_test:
        #src = os.path.join(train_class_path, img)
        #dst = os.path.join(test_class_path, img)
        #shutil.move(src, dst)

    #print(f"✅ Moved {test_count} images to test/{class_name}")

#print("\n🎯 80–20 split completed successfully!")


Class: bridge
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/bridge

Class: clean
Total images: 165
Train target: 132
Test target:  33
✅ Moved 33 images to test/clean

Class: cmp
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/cmp

Class: crack
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/crack

Class: ler
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/ler

Class: open
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/open

Class: vias
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/vias

Class: others
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/others

🎯 80–20 split completed successfully!


In [9]:
train_data = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=(224,224),
    batch_size=32,
    shuffle=True
)

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_PATH,
    image_size=(224,224),
    batch_size=32,
    shuffle=False
)


Found 1032 files belonging to 8 classes.
Found 264 files belonging to 8 classes.


In [10]:
import tensorflow as tf

# Reload only to get class names
temp_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/drive/MyDrive/Datasets/train",
    image_size=(224,224),
    batch_size=32
)

class_names = temp_ds.class_names
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("Num classes:", NUM_CLASSES)


Found 1032 files belonging to 8 classes.
Classes: ['bridge', 'clean', 'cmp', 'crack', 'ler', 'open', 'others', 'vias']
Num classes: 8


In [11]:
import os

for c in os.listdir(TRAIN_PATH):
    print(c, len(os.listdir(TRAIN_PATH + "/" + c)))


bridge 129
clean 129
cmp 129
crack 129
ler 129
open 129
vias 129
others 129


### Normalize + Add Data Augmentation

In [ ]:
from tensorflow.keras import layers

# Normalize (0–255 → 0–1)
normalization = layers.Rescaling(1./255)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
])



# Apply to datasets
train_data = train_data.map(lambda x, y: (normalization(data_augmentation(x)), y))
test_data  = test_data.map(lambda x, y: (normalization(x), y))


### Build the MobileNet Model (Your AI Brain)

In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models

base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(224,224,3),
    include_top=False,
    #weights="imagenet"   # ✅ IMPORTANT
)

# Freeze backbone
base_model.trainable = True  # ✅ IMPORTANT

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 576)            │         2,304 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,016,312 (3.88 MB)

 Trainable params: 1,003,048 (3.83 MB)

 Non-trainable params: 13,264 (51.81 KB)

##** Compile the Model**

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]   # ONLY accuracy
)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

y = []
for _, labels in train_data:
    y.extend(labels.numpy())

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

class_weights = dict(enumerate(class_weights))
print(class_weights)


{0: np.float64(0.7408088235294118), 1: np.float64(1.0494791666666667), 2: np.float64(1.325657894736842), 3: np.float64(0.9595238095238096), 4: np.float64(0.8995535714285714), 5: np.float64(0.8125), 6: np.float64(1.57421875), 7: np.float64(1.0833333333333333)}


### Train the Model

In [ ]:
EPOCHS = 15   # good for small dataset

history = model.fit(
    train_data,
    validation_data=train_data,
    epochs=15,
    class_weight=class_weights
)




Epoch 1/15


ValueError: Attr 'Toutput_types' of 'OptionalFromValue' Op passed list of length 0 less than minimum 1.

In [ ]:
model.save("/content/drive/MyDrive/Wafer-detect.keras")

